# Embeddings and Contrastive Learning: Making Similar Mean Close

The full code of [Embeddings and Contrastive Learning: Making Similar Mean Close](https://residual-stream.lvtan71.workers.dev/posts/embeddings-and-contrastive-learning/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/embeddings-and-contrastive-learning) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/embeddings-and-contrastive-learning.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 2 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "torch==2.14.*" "transformers==5.17.*" sentence-transformers

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "embeddings-and-contrastive-learning"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

In [ ]:
!tar xzf blog-47b01b1.tar.gz

## `word2vec.py`

Skip-gram with negative sampling on the training text; saves `word2vec.pt` and prints the loss, nearest neighbors and a few cosines. Takes a seed as an optional argument (default 0).

In [ ]:
!{sys.executable} word2vec.py

The post printed:

```text
60,355 words, 1,685 kept (5+ occurrences)
final loss 2.071 (at the start: (1 + k) log 2 = 4.16)
attention   -> sparse, multi-head, hybrid, transformer, flashattention, gqa
cache       -> kv, kv-cache-explained, linearly, per-sequence, quantization, iteration-level
gradient    -> descent, alive, mse, convex, curvature, failures
token       -> representation, accepted, dimensional, speculative, decoder, single
vietnamese  -> english, questions, route, reader, foundations, list
gpu         -> h100, datasheet, laptop, tflops, cores, bandwidth
cos(cache, caching) = 0.40, rank 71 of 1,685
cos(token, tokens) = 0.49, rank 14 of 1,685
cos(cache, cached) = 0.39, rank 89 of 1,685
```

## `more_seeds.py`

Runs `word2vec.py` with seeds 1 and 2 (saves `word2vec1.pt`, `word2vec2.pt`).

In [ ]:
!{sys.executable} more_seeds.py

The post printed:

```text
60,355 words, 1,685 kept (5+ occurrences)
final loss 2.037 (at the start: (1 + k) log 2 = 4.16)
attention   -> hybrid, multi-head, sparse, sliding, transformer, window
cache       -> kv, kv-cache-explained, bill, linearly, per-sequence, prefill
gradient    -> descent, alive, mse, failures, minimum, convex
token       -> moment, re-read, vocabulary, accepted, ch, representation
vietnamese  -> english, questions, keywords, rotary, settings, shown
gpu         -> h100, datasheet, synchronize, laptop, tflops, dense
cos(cache, caching) = 0.45, rank 46 of 1,685
cos(token, tokens) = 0.54, rank 13 of 1,685
cos(cache, cached) = 0.39, rank 103 of 1,685
60,355 words, 1,685 kept (5+ occurrences)
final loss 2.030 (at the start: (1 + k) log 2 = 4.16)
attention   -> hybrid, sparse, shipped, masked, multi-head, cross-attention
cache       -> kv, kv-cache-explained, hit, per-sequence, matched, vllm-architecture-explained
gradient    -> descent, mse, separable, reaching, alive, arriving
token       -> representation, contributes, re-read, character, wasted, decoder
vietnamese  -> english, questions, part, catch, combined, sends
gpu         -> h100, datasheet, bandwidth, tflop, ridge, kb
cos(cache, caching) = 0.50, rank 15 of 1,685
cos(token, tokens) = 0.48, rank 45 of 1,685
cos(cache, cached) = 0.49, rank 16 of 1,685
```

## `stable.py`

Which nearest neighbors appear in all three seeds, or in two.

In [ ]:
!{sys.executable} stable.py

The post printed:

```text
attention   in all 3 seeds: sparse, multi-head, hybrid;  in 2: transformer
cache       in all 3 seeds: kv, kv-cache-explained, per-sequence;  in 2: linearly
gradient    in all 3 seeds: descent, alive, mse;  in 2: convex, failures
token       in all 3 seeds: representation;  in 2: accepted, decoder, re-read
vietnamese  in all 3 seeds: english, questions;  in 2: -
gpu         in all 3 seeds: h100, datasheet;  in 2: laptop, tflops, bandwidth
```

## `analogies.py`

The "b - a + c lands near d" test on six analogies from the blog's vocabulary.

In [ ]:
!{sys.executable} analogies.py

The post printed:

```text
encoder : decoder :: input : ?   top answer 'trigger', the expected 'output' is ranked 121 of 1,682
keys : values :: key : ?   top answer 'self', the expected 'value' is ranked 36 of 1,682
prefill : decode :: prompt : ?   top answer 'character', the expected 'generation' is ranked 168 of 1,682
layer : layers :: token : ?   top answer 'speculative', the expected 'tokens' is ranked 91 of 1,682
head : heads :: weight : ?   top answer 'neurips', the expected 'weights' is ranked 15 of 1,682
key : keys :: value : ?   top answer 're-read', the expected 'values' is ranked 9 of 1,682
```

## `cosine_vs_dot.py`

Vector length against word frequency, and the nearest neighbors of "cache" by dot product and by cosine.

In [ ]:
!{sys.executable} cosine_vs_dot.py

The post printed:

```text
vector length vs log word frequency: correlation -0.20
nearest to 'cache' by dot product: kv (4.9), xhat (7.1), kv-cache-explained (5.4), nh (5.7), color-background (7.9), fill (7.4)
nearest to 'cache' by cosine     : kv (4.9), kv-cache-explained (5.4), linearly (4.6), per-sequence (4.9), quantization (3.8), iteration-level (4.9)
(vector lengths in brackets; the median word's is 4.3)
```

## `infonce.py`

InfoNCE on three 2-D pairs at four temperatures, and on random embeddings against log B.

In [ ]:
!{sys.executable} infonce.py

The post printed:

```text
cosine similarities (rows: a, columns: b):
   +0.949  +0.196  -0.995
   +0.316  +0.981  -0.100
   -0.868  +0.000  +0.956
temperature 1.0 : loss 0.5106
temperature 0.5 : loss 0.2332
temperature 0.1 : loss 0.0006
temperature 0.05: loss 0.0000
batch     4: random embeddings give loss 1.43 at temperature 1, 2.11 at 0.1; log B = 1.39
batch    64: random embeddings give loss 4.17 at temperature 1, 4.93 at 0.1; log B = 4.16
batch  1024: random embeddings give loss 6.94 at temperature 1, 7.68 at 0.1; log B = 6.93
```

## `contrastive.py`

Baselines, and a contrastive linear head on averaged word2vec vectors at batch 4, 16 and 64, three seeds each: same-section recall@1 on the held-out sections.

In [ ]:
!{sys.executable} contrastive.py

The post printed:

```text
156 training sections, 66 held-out sections
before training, the encoder is exactly the word2vec average: recall@1 0.152
random word vectors, averaged:   recall@1 0.061
word2vec vectors, averaged:      recall@1 0.152, 0.136, 0.136 (3 word2vec seeds)
contrastive, batch   4 ( 3 negatives): recall@1 0.197, 0.152, 0.212 (3 seeds); on training sections 0.121, 0.121, 0.121
contrastive, batch  16 (15 negatives): recall@1 0.167, 0.182, 0.212 (3 seeds); on training sections 0.136, 0.152, 0.136
contrastive, batch  64 (63 negatives): recall@1 0.212, 0.227, 0.212 (3 seeds); on training sections 0.182, 0.182, 0.167
```

## `pretrained.py`

all-MiniLM-L6-v2 and multilingual MiniLM-L12 on the same test, and on matching English sections to their Vietnamese translations.

In [ ]:
!{sys.executable} pretrained.py

The post printed:

```text
66 held-out English sections; 118 English/Vietnamese section pairs
averaged word2vec: average cosine between different held-out windows 0.81
all-MiniLM-L6-v2 (English): 23M parameters, reads up to 256 tokens
    same-section recall@1 0.318   English -> Vietnamese recall@1 0.576 (236 sections in 2.0 s)   average cosine between different windows 0.16
multilingual MiniLM-L12: 118M parameters, reads up to 128 tokens
    same-section recall@1 0.212   English -> Vietnamese recall@1 1.000 (236 sections in 2.0 s)   average cosine between different windows 0.29
```